<center><h1>Sun_Yizhi_HW3</h1></center>
<br>
<br>

Name: Yizhi Sun
<br>
Github Username: yizhi515
<br>
USC ID: 6129500198

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import glob
import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
data_path = "../data/AReM"

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

total_datasets = 0

for activity in activities:
    files = glob.glob(
        os.path.join(data_path, activity, "dataset*.csv")
    )
    print(f"{activity}: {len(files)} datasets")
    total_datasets += len(files)

print(f"\nTotal number of datasets: {total_datasets}")

bending1: 7 datasets
bending2: 6 datasets
cycling: 15 datasets
lying: 15 datasets
sitting: 15 datasets
standing: 15 datasets
walking: 15 datasets

Total number of datasets: 88


### (b) Test and Train Data

In [3]:
train_files = []
test_files = []

for activity in activities:
    files = glob.glob(
    os.path.join(data_path, activity, "dataset*.csv")
)
    
    # Sort files by dataset number
    files = sorted(
        files,
        key=lambda x: int(os.path.basename(x).replace("dataset", "").replace(".csv", ""))
    )
    
    for file in files:
        dataset_num = int(
            os.path.basename(file).replace("dataset", "").replace(".csv", "")
        )
        
        if activity in ["bending1", "bending2"]:
            if dataset_num in [1, 2]:
                test_files.append(file)
            else:
                train_files.append(file)
        else:
            if dataset_num in [1, 2, 3]:
                test_files.append(file)
            else:
                train_files.append(file)

print("Number of training instances:", len(train_files))
print("Number of test instances:", len(test_files))
print("Total instances:", len(train_files) + len(test_files))

Number of training instances: 69
Number of test instances: 19
Total instances: 88


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile (Q1)
- Third quartile (Q3)
- Range
- Interquartile range (IQR)
- Root mean square (RMS)
- Skewness
- Kurtosis

#### ii. Extraction

In [4]:
feature_names = [
    "min",
    "max",
    "mean",
    "median",
    "std",
    "1st_quart",
    "3rd_quart"
]

columns = ["Instance"]

for i in range(1, 7):
    for feature in feature_names:
        columns.append(f"{feature}{i}")

all_features = []
instance_num = 1

for activity in activities:
    files = glob.glob(
        os.path.join(data_path, activity, "dataset*.csv")
    )

    files = sorted(
        files,
        key=lambda x: int(
            os.path.basename(x)
            .replace("dataset", "")
            .replace(".csv", "")
        )
    )

    for file in files:
        data = pd.read_csv(
            file,
            skiprows=5,
            header=None,
            sep=r"\s*,\s*|\s+",
            engine="python",
            usecols=range(7)
        )

        instance_features = [instance_num]

        for i in range(1, 7):
            series = data.iloc[:, i]

            instance_features.extend([
                series.min(),
                series.max(),
                series.mean(),
                series.median(),
                series.std(),
                series.quantile(0.25),
                series.quantile(0.75)
            ])

        all_features.append(instance_features)
        instance_num += 1

features_df = pd.DataFrame(
    all_features,
    columns=columns
)

print(
    "Shape of extracted feature dataset:",
    features_df.shape
)

display(features_df)

Shape of extracted feature dataset: (88, 43)


,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [5]:
feature_data = features_df.drop(columns=["Instance"])

# Standard deviation of each feature
std_estimates = feature_data.std(ddof=1)

# Bootstrap 90% confidence intervals
np.random.seed(42)

n_bootstrap = 10000
bootstrap_stds = []

for _ in range(n_bootstrap):
    bootstrap_sample = feature_data.sample(
        n=len(feature_data),
        replace=True
    )
    
    bootstrap_stds.append(
        bootstrap_sample.std(ddof=1).values
    )

bootstrap_stds = np.array(bootstrap_stds)

# 90% percentile bootstrap confidence interval
lower_bounds = np.percentile(bootstrap_stds, 5, axis=0)
upper_bounds = np.percentile(bootstrap_stds, 95, axis=0)

std_results = pd.DataFrame({
    "Feature": feature_data.columns,
    "Standard Deviation": std_estimates.values,
    "90% CI Lower": lower_bounds,
    "90% CI Upper": upper_bounds
})

display(std_results)

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.569975,8.267172,10.781806
1,max1,4.394362,3.350842,5.299900
2,mean1,5.335718,4.704280,5.889400
3,median1,5.440054,4.786857,6.009184
4,std1,1.772153,1.565101,1.950318
5,1st_quart1,6.153590,5.554347,6.653597
6,3rd_quart1,5.138925,4.330202,5.844784
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.622054,5.402310
9,mean2,1.574164,1.396991,1.706516


#### iv. Select Features

I would select minimum, mean, and maximum as the three most important time-domain features.

- Minimum captures the lower extreme of the signal.
- Maximum captures the upper extreme of the signal.
- Mean summarizes the overall level of the signal.

Together, these features provide information about both the central tendency and the range of variation in each time series.

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS for the cubic regression is expected to be less than or equal to that of the linear regression.

The cubic model is more flexible and contains the linear model as a special case when the coefficients of \(X^2\) and \(X^3\) are zero. Therefore, the cubic model can fit the training data at least as well as the linear model, and possibly better by fitting some of the noise.

### (b) Linear Test

The linear regression is expected to have a lower test RSS than the cubic regression.

Since the true relationship between \(X\) and \(Y\) is linear, the linear model has the correct level of flexibility. The cubic model includes unnecessary \(X^2\) and \(X^3\) terms, which can fit noise in the training data and lead to overfitting. Therefore, the cubic model may have higher variance and worse performance on new test data.

### (c) Not Linear Train

The cubic regression is expected to have a training RSS less than or equal to that of the linear regression.

The cubic model is more flexible and contains the linear model as a special case. Therefore, it can fit the training data at least as well as the linear model, regardless of the true form of the relationship between \(X\) and \(Y\).

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS.

Although the true relationship is non-linear, we do not know how far it is from linear. The cubic model is more flexible and may reduce bias if the true relationship is strongly non-linear, but it also has higher variance. If the true relationship is only slightly non-linear, the linear model may still have a lower test RSS. Therefore, the result depends on the bias-variance tradeoff.

## 3. ISLR 3.7.3 - Extra Practice

### (a)

The correct answer is **(iii)**.

For fixed IQ and GPA, the difference in predicted salary between a college graduate and a high school graduate is

$$
35 - 10(\text{GPA}).
$$

If GPA is greater than 3.5, this difference becomes negative, meaning that high school graduates are predicted to earn more than college graduates. Therefore, high school graduates earn more provided that the GPA is high enough.

### (b)

For a college graduate with GPA = 4.0 and IQ = 110,

$$
X_4 = 4.0 \times 110 = 440,
$$

and

$$
X_5 = 4.0 \times 1 = 4.
$$

Therefore,

$$
\hat{Y}
= 50 + 20(4) + 0.07(110) + 35 + 0.01(440) - 10(4)
= 137.1.
$$

The predicted starting salary is **$137,100**.

### (c)

False.

The magnitude of the coefficient alone does not determine whether there is evidence of an interaction effect. The interaction variable $GPA \times IQ$ can take relatively large values, so even a coefficient of 0.01 may have a meaningful effect on the predicted response.

In addition, statistical evidence for an interaction should be assessed using quantities such as the standard error, t-statistic, p-value, or confidence interval, rather than the coefficient magnitude alone.

## 4. ISLR 3.7.5 - Extra Practice 

Since

$$
\hat{y}_i = x_i\hat{\beta}
$$

and

$$
\hat{\beta}
=
\frac{\sum_{i'=1}^{n} x_{i'}y_{i'}}
{\sum_{j=1}^{n} x_j^2},
$$

substituting \(\hat{\beta}\) into \(\hat{y}_i\) gives

$$
\hat{y}_i
=
x_i
\frac{\sum_{i'=1}^{n} x_{i'}y_{i'}}
{\sum_{j=1}^{n} x_j^2}.
$$

Distributing \(x_i\) inside the summation,

$$
\hat{y}_i
=
\sum_{i'=1}^{n}
\frac{x_i x_{i'}}
{\sum_{j=1}^{n} x_j^2}
y_{i'}.
$$

Therefore, comparing this with

$$
\hat{y}_i
=
\sum_{i'=1}^{n} a_{i'}y_{i'},
$$

we obtain

$$
\boxed{
a_{i'}
=
\frac{x_i x_{i'}}
{\sum_{j=1}^{n} x_j^2}
}.
$$

Thus, each fitted value \(\hat{y}_i\) is a linear combination of the observed response values \(y_1,\ldots,y_n\).